In [ ]:
import cv2
import glob
import os
import torch

from transformers import AutoModel
from tqdm import tqdm

In [ ]:
ROOT = "/mnt/stor/datasets/CBIS-DDSM/"

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

data_dir = os.path.join(ROOT, "png")
save_dir = os.path.join(ROOT, "cropped_png")
os.makedirs(save_dir, exist_ok=True)
os.makedirs(save_dir.replace("png", "masks"), exist_ok=True)

model = AutoModel.from_pretrained("ianpan/mammo-crop", trust_remote_code=True)
model = model.eval().to(device)

files = glob.glob(os.path.join(data_dir, "*.png"))
for f in tqdm(files):
    img = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
    img_shape = torch.tensor([img.shape[:2]]).to(device)
    x = model.preprocess(img)
    x = torch.from_numpy(x)
    x = x.expand(1, 1, -1, -1)
    x = x.to(device)
    with torch.inference_mode():
        y = model(x, img_shape)
    y = y[0].cpu().numpy()
    x, y, w, h = y
    cropped_img = img[y : y + h, x : x + w]
    pid = f.split("/")[-2]
    save_path = os.path.join(save_dir, os.path.basename(f))
    cv2.imwrite(save_path, cropped_img)
    mask = cv2.imread(f.replace("png/", "masks/"), cv2.IMREAD_GRAYSCALE)
    mask = mask[y : y + h, x : x + w]
    cv2.imwrite(save_path.replace("png/", "masks/"), mask)

In [ ]:
f.replace("png", "masks")

In [ ]:
import matplotlib.pyplot as plt


x = cv2.imread(save_path, 0)
y = cv2.imread(save_path.replace("png/", "masks/"), 0)

assert x.shape == y.shape 

plt.subplot(1, 2, 1)
plt.imshow(x, cmap="gray")
plt.subplot(1, 2, 2)
plt.imshow(y, cmap="gray")
plt.show()